<center><p float="center">
  <img src="https://images.pexels.com/photos/3184465/pexels-photo-3184465.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=2" width="720"/>
</p></center>

<center><font size=6>Employee Job Switch Prediction</font></center>

## Problem Statement

### Business Context

Ed-tech firms that run data-science bootcamps sign up a large pool of
trainees before deciding who gets a full-time offer. Out of this pool,
the company would like to know in advance who is genuinely interested
in joining them versus who will move on to a different employer once
the training wraps up. Knowing this in advance cuts down on training
spend and helps the company focus its hiring effort on the right
candidates. The goal here is to study what drives a trainee's decision
and to build a model that estimates the chance that a given candidate
will look for a new job rather than stay with the sponsoring company.

### Objective
The ed-tech company wants an estimate of how likely each candidate is
to search for a new job once training ends. Working as the data
scientist on this problem, the task is to explore the factors that
sway a candidate's decision and to build a model that outputs the
probability of a candidate looking for a new role, using the
candidate's profile, demographics and experience data.

### Data Description

* enrollee_id: Unique ID for the candidate
* city: City code
* city_development_index: Development index of the city (scaled)
* gender: Gender of the candidate
* relevent_experience: Relevant experience of the candidate
* enrolled_university: Type of university course enrolled if any
* education_level: Education level of candidate
* major_discipline: Education major discipline of the candidate
* experience: Candidate total experience in years
* company_size: No of employees in current employer's company
* company_type: Type of current employer
* lastnewjob: Difference in years between previous job and current job
* training_hours: training hours completed
* target: 0 - Not looking for a job change, 1 - Looking for a job change

## Importing Required Libraries

In [ ]:
# libraries for reading and working with tabular data
import numpy as np
import pandas as pd

# libraries for plotting
import matplotlib.pyplot as plt
import seaborn as sns

# utilities to tune, split and score a model
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

# utility to fill in missing values
from sklearn.impute import SimpleImputer

# logistic regression classifier
from sklearn.linear_model import LogisticRegression

# utilities to balance the classes
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# to keep the notebook output clean
import warnings

warnings.filterwarnings("ignore")


## Loading the Data

In [ ]:
jobs_df = pd.read_csv("jobs_data.csv")

## First Look at the Data

Before doing anything else with a new dataset it helps to:
- peek at a handful of rows to confirm the file loaded the way we expect
- note how many rows/columns we are working with
- check that every column's dtype matches what it should hold
- run a quick statistical summary of the numeric fields

### First Few Records

In [ ]:
# preview the first 5 records
jobs_df.head()

### Size of the Data

In [ ]:
# rows and columns present in the data
jobs_df.shape

* The data holds 14 attributes across 19158 candidate records.

### Column Data Types

In [ ]:
# dtype and null-count overview
jobs_df.info()

* Only 5 columns hold numeric values, the rest are stored as objects.
* 8 columns contain fewer than 19158 non-null entries, i.e. they carry missing values.

### Summary Statistics

In [ ]:
jobs_df.describe(include="all").T

**Observations**
* `city_development_index`: this is already a normalized score, and an average of roughly 0.82 tells us that a large share of records come from well-developed (metro) cities. The score itself ranges from about 0.448 up to 0.949, so there is still a fair spread.
* `training_hours`: values range widely, from 60 up to 336 hours, averaging around 65 hours; a sizeable chunk of trainees log 88 hours or less.
* `target`: the classes are skewed - close to 75% of candidates are not job hunting, suggesting many just wanted to pick up new skills.

In [ ]:
# summary of the non-numeric (categorical) columns
jobs_df.describe(exclude=np.number).T

**Observations**
* city_103 shows up more than any other city code.
* Male candidates dominate the gender column.
* Most candidates report having some relevant work experience already.
* The majority were not enrolled in any university course.
* Graduates make up the largest education-level group.
* STEM is the most common major discipline.
* Pvt Ltd is the most frequent employer type.

In [ ]:
# frequency counts for every column
column_list = jobs_df.columns

for col in column_list:
    print(jobs_df[col].value_counts())
    print("-" * 50)

### Duplicate Rows

In [ ]:
# checking for exact duplicate rows
jobs_df.duplicated().sum()

* No duplicate rows are present in the dataset.

### Missing Values

In [ ]:
# percentage of missing entries per column
round(jobs_df.isnull().sum() / jobs_df.isnull().count() * 100, 2)

* `company_type` is missing in 32.05% of rows.
* `company_size` is missing in 30.99% of rows.
* `gender` is missing in 23.53% of rows.
* `major_discipline` is missing in 14.68% of rows.
* `education_level` is missing in 2.40% of rows.
* `last_new_job` is missing in 2.21% of rows.
* `enrolled_university` is missing in 2.01% of rows.
* `experience` is missing in 0.34% of rows.
* Imputation will be handled after the train/validation/test split.

## <a name='eda_summary'>Exploratory Data Analysis - Key Findings</a>

### **Note**: The full step-by-step EDA has been done in previous case studies, so here we call out only the headline observations. The complete walkthrough is kept in the <a href='#eda_appendix'>appendix</a> at the end of this notebook.

**A few helper functions are defined below to drive the plots used through this analysis.**

In [ ]:
# plots a histogram stacked on top of a boxplot for one numeric column


def plot_num_distribution(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Draws a boxplot above a histogram sharing the same x-axis

    data: dataframe
    feature: column to plot
    figsize: figure size (default (12,7))
    kde: draw a density curve on the histogram (default False)
    bins: number of histogram bins (default None)
    """
    fig, (box_ax, hist_ax) = plt.subplots(
        nrows=2,
        sharex=True,
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )
    sns.boxplot(
        data=data, x=feature, ax=box_ax, showmeans=True, color="violet"
    )  # the boxplot also marks the mean with a triangle
    sns.histplot(
        data=data, x=feature, kde=kde, ax=hist_ax, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=hist_ax
    )
    hist_ax.axvline(data[feature].mean(), color="green", linestyle="--")  # mean marker
    hist_ax.axvline(data[feature].median(), color="black", linestyle="-")  # median marker


In [ ]:
# bar chart with the count (or %) printed above each bar


def plot_category_counts(data, feature, perc=False, n=None):
    """
    Countplot annotated with the value on top of each bar

    data: dataframe
    feature: categorical column to plot
    perc: annotate with percentage instead of raw count (default False)
    n: keep only the top n categories (default None -> keep all)
    """

    n_rows = len(data[feature])
    n_levels = data[feature].nunique()
    width = (n_levels if n is None else n) + 2
    plt.figure(figsize=(width, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        hue=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for bar in ax.patches:
        if perc:
            label = "{:.1f}%".format(100 * bar.get_height() / n_rows)
        else:
            label = bar.get_height()

        x_pos = bar.get_x() + bar.get_width() / 2
        y_pos = bar.get_height()

        ax.annotate(
            label,
            (x_pos, y_pos),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.show()


In [ ]:
# stacked bar chart showing how the target splits across a categorical predictor


def plot_target_share(data, predictor, target):
    """
    Prints a cross-tab and plots a normalized stacked bar chart

    data: dataframe
    predictor: independent categorical column
    target: dependent column
    """
    n_levels = data[predictor].nunique()
    order_by = data[target].value_counts().index[-1]

    counts_tab = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=order_by, ascending=False
    )
    print(counts_tab)
    print("-" * 120)

    pct_tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=order_by, ascending=False
    )
    pct_tab.plot(kind="bar", stacked=True, figsize=(n_levels + 1, 5))
    plt.legend(loc="lower left", frameon=False)
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()


In [ ]:
# distribution of a numeric predictor split by the two target classes


def plot_feature_vs_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    classes = data[target].unique()

    axs[0, 0].set_title("Distribution for target=" + str(classes[0]))
    sns.histplot(
        data=data[data[target] == classes[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
    )

    axs[0, 1].set_title("Distribution for target=" + str(classes[1]))
    sns.histplot(
        data=data[data[target] == classes[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
    )

    axs[1, 0].set_title("Boxplot vs target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (outliers hidden) vs target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()


### Univariate Analysis

#### `target`

In [ ]:
plot_category_counts(jobs_df, "target", perc=True)

* The target classes are imbalanced - close to 75% of candidates are not looking for a job change.

#### `city_development_index`

In [ ]:
plot_num_distribution(jobs_df, "city_development_index")

* The distribution leans left (negatively skewed).
* Records with an index below ~0.45 show up as outliers - these are likely tier-3 or under-developed cities.
* Worth a closer look.

In [ ]:
jobs_df[jobs_df["city_development_index"] < 0.45]

* Every record below 0.45 belongs to city_33, so this looks like a genuine pattern rather than noise - no outlier treatment needed.

#### `training_hours`

In [ ]:
plot_num_distribution(jobs_df, "training_hours")

* Right-skewed with a long tail of outliers.
* Anything above roughly 175 hours shows up as an outlier on the boxplot.

#### `gender`

In [ ]:
plot_category_counts(jobs_df, "gender")

* About 69% of candidates identify as male.

#### `relevent_experience`

In [ ]:
plot_category_counts(jobs_df, "relevent_experience")

* Roughly 72% of candidates already have some relevant experience.

#### `company_size`

In [ ]:
plot_category_counts(jobs_df, "company_size")

* 16.1% worked at companies of 50-99 employees, and 13.4% at companies of 100-500 employees - the two largest buckets.

### Bivariate Analysis

In [ ]:
sns.pairplot(data=jobs_df, diag_kind="kde")
plt.show()

* For readability in the plots below we relabel the target as 'yes'/'no' instead of 1/0.

In [ ]:
jobs_df["target"] = jobs_df["target"].replace(1, "yes")
jobs_df["target"] = jobs_df["target"].replace(0, "no")

#### `target vs gender`

In [ ]:
plot_target_share(jobs_df, "gender", "target")

* Female candidates, followed by candidates of other genders, show up as the ones most actively looking for a change.
* Let's fold the missing values into an explicit 'unknown' category and re-check.

In [ ]:
jobs_df["gender"] = jobs_df["gender"].replace(np.nan, "unknown")

In [ ]:
plot_target_share(jobs_df, "gender", "target")

* Once grouped this way, the 'unknown' gender bucket has the highest share of candidates job-hunting.

#### `target vs relevent_experience`

In [ ]:
plot_target_share(jobs_df, "relevent_experience", "target")

* 35% of candidates without relevant experience are job-hunting - likely freshers seeking their first real opportunity.
* 20% of candidates who do have relevant experience are also looking, possibly after upskilling for a new role.

#### `target vs company_type`

In [ ]:
plot_target_share(jobs_df, "company_type", "target")

* Candidates from newly-funded startups show the least interest in switching, perhaps because there is growth potential right where they are.
* Across most other employer types, roughly 20% are on the lookout for a new role.

#### `target vs last_new_job`

In [ ]:
plot_target_share(jobs_df, "last_new_job", "target")

* Our earlier guess about 'never' representing freshers/unemployed candidates holds up - this group shows the highest interest in a job change.

#### `target vs training_hours`

In [ ]:
plot_feature_vs_target(jobs_df, "training_hours", "target")

* Training hours don't appear to move the needle on the target.

#### `target vs city_development_index`

In [ ]:
plot_feature_vs_target(jobs_df, "city_development_index", "target")

* There's a clear separation between the two target groups here.
* Candidates from more developed cities are less likely to look for a change.
* Candidates from cities with a lower development index may be upskilling in hopes of relocating to bigger metros.

## Data Preprocessing

* `enrollee_id` is just a unique row identifier and carries no predictive signal, so it can be dropped.

In [ ]:
# enrollee_id is unique per row and adds nothing for modeling
jobs_df.drop(["enrollee_id"], axis=1, inplace=True)

### Encoding Categorical Columns

In [ ]:
jobs_df["city"].nunique()

* The `city` column has 123 distinct codes - we can shrink this down to 3 buckets:
  * Developed - city_development_index above 0.90
  * Developing - city_development_index between 0.74 and 0.90
  * Under-Developed - city_development_index between 0.4 and 0.74

In [ ]:
jobs_df["city_development_index"].describe()

In [ ]:
# bucket the index into quantile-based groups
jobs_df["city"] = pd.qcut(
    jobs_df["city_development_index"],
    q=[0, 0.25, 0.5, 1],
    labels=["Under_Developed", "Developing", "Developed"],
)

In [ ]:
jobs_df["city"].value_counts()

* The 123 city codes now collapse into 3 buckets.

In [ ]:
jobs_encoded = jobs_df.copy()

In [ ]:
jobs_encoded.head()

* Missing-value imputation will happen after the train/validation/test split to keep the sets independent of each other.

### Preparing Data for Modeling

In [ ]:
features = jobs_encoded.drop(["target"], axis=1)
target = jobs_encoded["target"].apply(lambda x: 1 if x == "yes" else 0)

* `pandas` may store missing entries as float NaN or as the literal string "unknown".
* We standardize both to `pd.NA` before splitting.

In [ ]:
# standardize missing markers to pandas NA
features.fillna(pd.NA, inplace=True)
features.replace(to_replace="unknown", value=pd.NA, inplace=True)

In [ ]:
# first carve out a held-out test set
feats_temp, feats_test, tgt_temp, tgt_test = train_test_split(
    features, target, test_size=0.2, random_state=1, stratify=target
)

# then split the remainder into train and validation
feats_train, feats_val, tgt_train, tgt_val = train_test_split(
    feats_temp, tgt_temp, test_size=0.25, random_state=1, stratify=tgt_temp
)
print(feats_train.shape, feats_val.shape, feats_test.shape)

In [ ]:
print("Rows in training data =", feats_train.shape[0])
print("Rows in validation data =", feats_val.shape[0])
print("Rows in test data =", feats_test.shape[0])

* We keep hold of the row indices for each split so we can pull the right rows back out after encoding.

In [ ]:
idx_train = set(feats_train.index)
idx_val = set(feats_val.index)
idx_test = set(feats_test.index)

### Handling Missing Values

In [ ]:
feats_train.isnull().sum()

In [ ]:
feats_val.isnull().sum()

In [ ]:
feats_test.isnull().sum()

* The columns ***gender, enrolled_university, education_level, major_discipline, experience, company_size, company_type,*** and **last_new_job** all carry missing values.
* All of these are categorical, so the mode is the right statistic to impute with.
    * The mode fills gaps with the most frequently occurring category, keeping the imputed values consistent with the rest of the data.

In [ ]:
# imputer that fills gaps with the most frequent category
mode_imputer = SimpleImputer(missing_values=pd.NA, strategy="most_frequent")

* To avoid leaking information from validation/test into training, we learn the fill value on the training set only.
* `fit_transform()` learns the value and fills the training data; `transform()` reuses that learned value elsewhere.

In [ ]:
# learn from the training data and fill it
feats_train[:] = mode_imputer.fit_transform(feats_train[:])

# reuse the same learned values on validation and test
feats_val[:] = mode_imputer.transform(feats_val[:])
feats_test[:] = mode_imputer.transform(feats_test[:])

### One-Hot Encoding

In [ ]:
# stack all three splits together before encoding
combined_feats = pd.concat([feats_train, feats_val, feats_test])

In [ ]:
# one-hot encode the categorical columns
combined_feats = pd.get_dummies(combined_feats, drop_first=True, dtype=int)
print(combined_feats.shape)

* Encoding expands the data out to 56 columns.

* Using the indices saved earlier, we can split the encoded frame back into train, validation and test.

In [ ]:
feats_train = combined_feats.loc[list(idx_train)]

feats_val = combined_feats.loc[list(idx_val)]

feats_test = combined_feats.loc[list(idx_test)]

## Model Building

### Choosing the Right Evaluation Metric

**Two ways the model can get it wrong:**
1. Flagging a candidate as job-hunting when they're actually staying - wasted recruiting effort.
2. Flagging a candidate as staying when they're actually job-hunting - a missed opportunity.

**Which mistake hurts more?**
* Missing a candidate who is genuinely job-hunting, since the HR team would then fail to reach out to someone who was actually reachable.

**How do we cut down on that kind of mistake (false negatives)?**
* We want Recall to be as high as possible - a higher Recall means fewer false negatives.

In [ ]:
# computes accuracy, recall, precision and f1 for a fitted sklearn classifier
def get_classification_metrics(model, X, y):
    """
    Returns a one-row dataframe of classification metrics

    model: fitted classifier
    X: feature matrix
    y: true labels
    """

    preds = model.predict(X)

    acc = accuracy_score(y, preds)
    recall = recall_score(y, preds)
    precision = precision_score(y, preds)
    f1 = f1_score(y, preds)

    metrics_df = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )

    return metrics_df


In [ ]:
# plots a confusion matrix with counts and percentages annotated
def plot_confusion_matrix(model, X, y):
    """
    model: fitted classifier
    X: feature matrix
    y: true labels
    """
    preds = model.predict(X)
    cm = confusion_matrix(y, preds)
    cell_labels = np.asarray(
        [
            ["{0:0.0f}".format(val) + "\n{0:.2%}".format(val / cm.flatten().sum())]
            for val in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=cell_labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")


### Baseline Model on the Raw Split

In [ ]:
base_model = LogisticRegression(random_state=1)
base_model.fit(feats_train, tgt_train)

In [ ]:
# training performance
base_train_perf = get_classification_metrics(base_model, feats_train, tgt_train)
print("Training performance:")
base_train_perf

In [ ]:
plot_confusion_matrix(base_model, feats_train, tgt_train)

In [ ]:
# validation performance
base_val_perf = get_classification_metrics(base_model, feats_val, tgt_val)
print("Validation performance:")
base_val_perf

In [ ]:
plot_confusion_matrix(base_model, feats_val, tgt_val)

- The model predicts the majority class (0) for every row.
- Precision, recall and F1 all come out to 0 as a result.
- Let's dig into why.

In [ ]:
tgt_train.value_counts()

* `tgt_train` mirrors the original imbalance - about 75% zeros and the rest ones.
   - This is the same skew we already saw in the overall target column.

* With recall stuck at zero, oversampling the minority class is worth a try.

**Cross-checking with Stratified K-Fold**

- K-fold cross-validation splits the data into k consecutive folds and rotates which fold is held out for validation while the rest form the training set.
- Stratified K-fold keeps the class ratio consistent across every fold.

In [ ]:
eval_metric = "recall"
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_scores_base = cross_val_score(
    estimator=base_model, X=feats_train, y=tgt_train, scoring=eval_metric, cv=cv_splitter
)
cv_scores_base

In [ ]:
plt.boxplot(cv_scores_base)
plt.show()

* Cross-validated recall on the training folds also comes out to zero.

### Model on Oversampled Data

In [ ]:
print("Before oversampling, count of 'Yes': {}".format(sum(tgt_train == 1)))
print("Before oversampling, count of 'No': {} \n".format(sum(tgt_train == 0)))

smote_sampler = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=1)
feats_train_os, tgt_train_os = smote_sampler.fit_resample(feats_train, tgt_train)

print("After oversampling, count of 'Yes': {}".format(sum(tgt_train_os == 1)))
print("After oversampling, count of 'No': {} \n".format(sum(tgt_train_os == 0)))

print("After oversampling, shape of features: {}".format(feats_train_os.shape))
print("After oversampling, shape of target: {} \n".format(tgt_train_os.shape))

In [ ]:
model_oversampled = LogisticRegression(random_state=1)

# fit on the oversampled training data
model_oversampled.fit(feats_train_os, tgt_train_os)

In [ ]:
os_train_perf = get_classification_metrics(model_oversampled, feats_train_os, tgt_train_os)
print("Training performance:")
os_train_perf

In [ ]:
os_val_perf = get_classification_metrics(model_oversampled, feats_val, tgt_val)
print("Validation performance:")
os_val_perf

In [ ]:
plot_confusion_matrix(model_oversampled, feats_val, tgt_val)

**Cross-checking with Stratified K-Fold**

- Same rotating-fold idea as before, applied to the oversampled training set.

In [ ]:
eval_metric = "recall"
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_scores_os = cross_val_score(
    estimator=model_oversampled, X=feats_train_os, y=tgt_train_os, scoring=eval_metric, cv=cv_splitter
)
cv_scores_os

In [ ]:
plt.boxplot(cv_scores_os)
plt.show()

* Training-fold recall now lands between roughly 0.63 and 0.64 - a clear step up from the baseline.

* That gain doesn't carry over to validation, which points to overfitting.
* Let's try regularization to rein that in.

### Regularization

In [ ]:
# base estimator to tune
lr_to_tune = LogisticRegression(random_state=1, solver="saga")

# candidate values for the regularization strength
param_grid = {"C": np.arange(0.1, 1.1, 0.1)}

# search for the best C using recall as the scoring metric
search = GridSearchCV(lr_to_tune, param_grid, scoring="recall")
search = search.fit(feats_train_os, tgt_train_os)

# grab the best estimator found
tuned_model = search.best_estimator_

# refit on the oversampled data
tuned_model.fit(feats_train_os, tgt_train_os)

In [ ]:
tuned_train_perf = get_classification_metrics(tuned_model, feats_train_os, tgt_train_os)
print("Training performance:")
tuned_train_perf

In [ ]:
tuned_val_perf = get_classification_metrics(tuned_model, feats_val, tgt_val)
print("Validation performance:")
tuned_val_perf

* Regularization trims the overfitting somewhat, but the model still isn't good enough to rely on.

### Model on Undersampled Data

In [ ]:
undersampler = RandomUnderSampler(random_state=1, sampling_strategy=1)
feats_train_us, tgt_train_us = undersampler.fit_resample(feats_train, tgt_train)

print("Before undersampling, count of '1': {}".format(sum(tgt_train == 1)))
print("Before undersampling, count of '0': {} \n".format(sum(tgt_train == 0)))

print("After undersampling, count of '1': {}".format(sum(tgt_train_us == 1)))
print("After undersampling, count of '0': {} \n".format(sum(tgt_train_us == 0)))

print("After undersampling, shape of features: {}".format(feats_train_us.shape))
print("After undersampling, shape of target: {} \n".format(tgt_train_us.shape))

In [ ]:
model_undersampled = LogisticRegression(random_state=1)
model_undersampled.fit(feats_train_us, tgt_train_us)

In [ ]:
us_train_perf = get_classification_metrics(model_undersampled, feats_train_us, tgt_train_us)
print("Training performance:")
us_train_perf

In [ ]:
us_val_perf = get_classification_metrics(model_undersampled, feats_val, tgt_val)
print("Validation performance:")
us_val_perf

* This version generalizes reasonably well between training and validation.

**Cross-checking with Stratified K-Fold**

- Same rotating-fold check, this time on the undersampled training set.

In [ ]:
eval_metric = "recall"
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_scores_us = cross_val_score(
    estimator=model_undersampled, X=feats_train_us, y=tgt_train_us, scoring=eval_metric, cv=cv_splitter
)
cv_scores_us

In [ ]:
plt.boxplot(cv_scores_us)
plt.show()

* Training-fold recall settles between about 0.5 and 0.52 - better than the very first (unbalanced) model.

## Comparing Models and Picking a Final One

In [ ]:
# side-by-side comparison of training performance

train_comparison_df = pd.concat(
    [
        base_train_perf.T,
        os_train_perf.T,
        tuned_train_perf.T,
        us_train_perf.T,
    ],
    axis=1,
)
train_comparison_df.columns = [
    "Logistic Regression",
    "Logistic Regression (oversampled)",
    "Logistic Regression (regularized)",
    "Logistic Regression (undersampled)",
]
print("Training performance comparison:")
train_comparison_df

In [ ]:
# side-by-side comparison of validation performance

val_comparison_df = pd.concat(
    [
        base_val_perf.T,
        os_val_perf.T,
        tuned_val_perf.T,
        us_val_perf.T,
    ],
    axis=1,
)
val_comparison_df.columns = [
    "Logistic Regression",
    "Logistic Regression (oversampled)",
    "Logistic Regression (regularized)",
    "Logistic Regression (undersampled)",
]
print("Validation performance comparison:")
val_comparison_df

* The very first model, trained on the raw imbalanced data, started at a recall of 0. Once we addressed the imbalance and applied regularization, recall climbed as high as 0.52.

* Of all the variants tried, the undersampled model comes out ahead.

* Since it also generalizes best from training to validation with the strongest recall, we'll carry the undersampled logistic regression forward as our final model.

### Performance on the Test Set

**With a final model chosen, let's see how it holds up on data it has never seen.**

In [ ]:
# performance on the held-out test set
us_test_perf = get_classification_metrics(model_undersampled, feats_test, tgt_test)
print("Test performance:")
us_test_perf

In [ ]:
plot_confusion_matrix(model_undersampled, feats_test, tgt_test)

* The undersampled model carries its generalized performance through to the test set.

### Reading the Coefficients

In [ ]:
# coefficients and intercept from the final logistic regression model

coeff_table = pd.DataFrame(
    np.append(model_undersampled.coef_, model_undersampled.intercept_),
    index=feats_train.columns.tolist() + ["Intercept"],
    columns=["Coefficients"],
)
coeff_table.T

* Logistic regression coefficients are expressed in log-odds, so we exponentiate them to get back to plain odds.
* **odds = exp(b)**
* The % change in odds works out to **(exp(b) - 1) * 100**

In [ ]:
# convert log-odds coefficients into odds
odds_vals = np.exp(model_undersampled.coef_[0])

# percentage change in odds
pct_change_odds = (np.exp(model_undersampled.coef_[0]) - 1) * 100

# show every column when printing
pd.set_option("display.max_columns", None)

odds_table = pd.DataFrame(
    {"Odds": odds_vals, "Change_odd%": pct_change_odds}, index=feats_train.columns
).T

odds_table

* `city_development_index`: with everything else held constant, a one-unit rise in this index multiplies the odds of job-hunting by roughly 1.05x - about a 5.5% increase.

* `training_hours`: holding everything else fixed, a one-unit rise in training hours multiplies the odds of job-hunting by roughly 0.99x - about a 0.08% decrease.

* `city - developing/developed`:
    * Candidates from developing cities have roughly 1.10x the odds of job-hunting compared with candidates from under-developed cities (~10.7% higher), keeping under-developed as the reference.
    * Candidates from developed cities have roughly 1.01x the odds compared with under-developed cities (~1.03% higher).

* `gender - male/other` - Male candidates show about 1.16% higher odds of job-hunting than female candidates, while candidates of other genders show about 0.52% lower odds than female candidates (female as reference).

* `relevent_experience - No relevant experience`: candidates without relevant experience have roughly 1.14x the odds of job-hunting versus those with relevant experience (~14.69% higher).

**The remaining coefficients can be read in the same way.**

## Conclusions and Recommendations for the Business

* Training hours barely move the needle on a candidate's likelihood of switching jobs, so the company shouldn't weight this attribute heavily during hiring decisions.
* Candidates from under-developed cities show the strongest pull toward switching - if such a candidate otherwise fits the role, they should be prioritized over candidates from developed/developing cities.
* Hiring across a wider range of gender identities would also help diversity goals, since non-binary/other-gender candidates show a higher likelihood of job-hunting.
* Freshers or candidates without relevant experience are more likely to be job-hunting - if they meet the role's bar, they're a good group to focus recruiting on.
* Candidates coming from early-stage startups, the public sector, or NGOs lean more toward switching than those from private companies or funded startups.
* Company size matters too - candidates from very large companies (1000-5000 or 10000+ employees) show up more among job-hunters, possibly seeking management roles or simply more comfortable moving given their broader exposure.

#### **Note**: This notebook focuses on the Week 1 model-tuning concepts. Additional models and hyperparameter searches could push these results further.

## <a name='eda_appendix'>Appendix: Full Exploratory Data Analysis</a>

### Univariate Analysis (continued)

#### `enrolled_university`

In [ ]:
plot_category_counts(jobs_df, "enrolled_university")

* 72.1% of candidates had no university enrollment, and 19.6% were enrolled full-time.

#### `education_level`

In [ ]:
plot_category_counts(jobs_df, "education_level")

* 60.5% of candidates are graduates, followed by 22.8% holding a Master's degree.

#### `major_discipline`

In [ ]:
plot_category_counts(jobs_df, "major_discipline")

* 75.5% of candidates studied a STEM discipline (science, technology, engineering or mathematics).

#### `company_type`

In [ ]:
plot_category_counts(jobs_df, "company_type")

* 51.2% of candidates worked at private companies, and 8.3% at startups.

#### `last_new_job`

In [ ]:
plot_category_counts(jobs_df, "last_new_job")

* 42% of candidates have a 1-year gap between their previous and current job, and 17.2% have a gap of more than 4 years.
* 'Never' can mean either a fresher or someone not currently employed.

### Bivariate Analysis (continued)

#### Correlation Check

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(jobs_df.corr(numeric_only=True), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral")
plt.show()

* No independent variable correlates strongly with the target or with each other.
* City development index shows a negative relationship with the target.

#### `target vs city`

In [ ]:
plot_target_share(jobs_df, "city", "target")

* Cities with the highest share of job-hunters are likely tier-2/tier-3 cities where candidates want better opportunities.
* This column separates the two target groups clearly enough to be a useful predictor.

#### `target vs major_discipline`

In [ ]:
plot_target_share(jobs_df, "major_discipline", "target")

* Every discipline shows a substantial share of candidates job-hunting.

#### `target vs experience`

In [ ]:
plot_target_share(jobs_df, "experience", "target")

* Candidates with under 10 years of experience make up most of the job-hunters.
* The '<1' bucket likely represents freshers hoping to land a role in their trained field.

#### `target vs company_size`

In [ ]:
plot_target_share(jobs_df, "company_size", "target")

* Roughly 20% of candidates are job-hunting regardless of the size of company they came from.

#### `target vs enrolled_university`

In [ ]:
plot_target_share(jobs_df, "enrolled_university", "target")

* All three enrollment categories show a large share of job-hunters.
* About 40% of full-time students are job-hunting, versus about 30% of part-time students.

#### `target vs education_level`

In [ ]:
plot_target_share(jobs_df, "education_level", "target")

* Graduates, Master's holders and high-schoolers all show a notable share of job-hunters.
* About 30% of graduates and 20% of Master's holders are looking for a change.

### To jump back to the summary section, click <a href='#eda_summary'>here</a>.